<hr>

## 🧰 INSTALLs


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: red;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

In [ ]:
# install numpy and pandas if not already installed
#!pip install numpy pandas

# install xgboost if not already installed
#!pip install xgboost

# install shap if not already installed
#!pip install shap

# install imbalanced-learn
#!pip install imbalanced-learn

<hr>

## 📂 IMPORTs


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: red;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

In [ ]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import roc_auc_score, f1_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

from xgboost import XGBClassifier

plt.style.use('ggplot')
pd.set_option('display.max_columns', 200)

<hr>

# 1️⃣ DATA PREPROCESSING


<style>
h1 {
    text-align: left;
    color: green;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

```text
1. Data Preparation for Machine Learning (data_cleaning handle invalid values, nulls, duplicates, outliers and DROP IDs, Names, and future info columns)
2. Train-Test SPLIT (80% Train / 20% Test) using stratify to keep same balance
3. Feature Engineering & Feauture Scaling (Cat/Num)
4. Check Data Skewness
```

## **🔻 STEP 0 — LOAD DATA**

##  **📝 STEP 1 — DATA PREPARATION**

### **DROP COLUMNS**
DROP COLUMNS We don't need for machine learning predict property_value based on other features!
- ID
- Names

columns to drop:
- `Identifiant de document`
- `Reference document`
- `1 Articles CGI`
- `2 Articles CGI`
- `3 Articles CGI`
- `4 Articles CGI`
- `5 Articles CGI`
- `Identifiant local`

In [ ]:
from pathlib import Path
import pandas as pd

INPUT_FILE = Path("../data/processed/CLEAN_ValeursFoncieres.csv")
OUTPUT_FILE = Path("../data/processed/ANALYSIS_DROP_ValeursFoncieres.csv")

CHUNK_SIZE = 500_000

cols_to_drop = [
    "Identifiant de document",
    "Reference document",
    "1 Articles CGI",
    "2 Articles CGI",
    "3 Articles CGI",
    "4 Articles CGI",
    "5 Articles CGI",
    "Identifiant local",
]

def drop_columns_chunked(input_file: Path, output_file: Path, cols_to_drop: list[str], chunk_size: int = 50_000) -> None:
    output_file.parent.mkdir(parents=True, exist_ok=True)

    first_write = True
    total_rows = 0

    for chunk_idx, chunk in enumerate(
        pd.read_csv(input_file, chunksize=chunk_size, dtype=str),
        start=1
    ):
        existing_cols_to_drop = [col for col in cols_to_drop if col in chunk.columns]
        chunk = chunk.drop(columns=existing_cols_to_drop)

        chunk.to_csv(
            output_file,
            mode="w" if first_write else "a",
            index=False,
            header=first_write,
            encoding="utf-8"
        )

        first_write = False
        total_rows += len(chunk)

        print(f"Chunk {chunk_idx} written -> {len(chunk):,} rows | total: {total_rows:,}")

    print("\n✅ Done! Dropped columns")
    print(f"Saved file: {output_file}")
    print(f"Total rows written: {total_rows:,}")

drop_columns_chunked(INPUT_FILE, OUTPUT_FILE, cols_to_drop, CHUNK_SIZE)

### **Parse DATE to YEAR & MONTH (CAT Variables)**

In [ ]:
# save file as ML_ValeursFoncieres.csv	

### **LOG TRANSFORM TARGET**

## **✂️ STEP 2 — Train/Test SPLIT (80/20%) Stratified**

**When Performing Train/Test Split:**
- Split Valeurs Foncieres dataset into **Train** and **Test** sets (80/20%) 
- Use **stratify by target variable** to maintain class distribution
- Use `random_state = 42`
- Consider stratified k-fold cross-validation later

✔ **To prevent data leakage, SPLIT BEFORE:**
- Feature Engineering (Encoding/Scaling)
- SMOTE
- Feature selection



### **DEFINE target (y) & features (X)**


In [ ]:
from duckdb import df
import pandas as pd

X = df.drop(columns=["property_value"]) # features: all columns except "property_value"
y = df["property_value"] # target variable

print(f"Features=(X) and target=(y) have been separated successfully! \nShapes:\nX =", X.shape, ", y =", y.shape)

### **SPLIT Train/Test sets**


- Train set: X_train AND y_train
- Test set:  X_test  AND y_test

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# ----------------------------
# Create bins for stratification (regression trick)
# ----------------------------
bins = pd.qcut(y, q=10, duplicates="drop")

# ----------------------------
# Train/Test split
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,                  # features
    y,                  # target variable
    test_size=0.2,      # 20% test set, 80% train set
    random_state=42,    # for reproducibility
    stratify=bins       # bins are used for stratification to ensure similar distribution of the target variable in both sets
)

# ----------------------------
# Save datasets
# ----------------------------
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

# ----------------------------
# Logging
# ----------------------------
print(f"Train/Test split completed successfully!")
print(100*"-")

print(f"\nTrain set Shapes:\nX_train = {X_train.shape}, y_train = {y_train.shape}")
display(X_train.head())

print(100*"-")

print(f"\nTest set Shapes: \nX_test = {X_test.shape}, y_test = {y_test.shape}")
display(X_test.head())

## **👷 STEP 3 — Feature Engineering & Scaling (Cat/Num)**

Steps to Features Engineering:
1. Identify Categorical vs Numerical Features
2. Encode Categorical Features & Scale Numerical Features
- Categorical Features:
    - Hot-one encoding for seperate categories
    - Labeling for ordinal variables
- Numeric Features:
    - Scaling because models can be sensitive to big numbers
3. Convert dtype to int or float
4. Save `X_train.csv`,`X_test.csv` and `y_train.csv`,`y_test.csv`

### **Identify Variables - Categorical vs Numeric**

In [ ]:
# categorical variables
cat_cols = [
    "transaction_type",     # OneHotEncode
    "street_number",        # OneHotEncode
    "btq_code",             # OneHotEncode
    "street_type",          # OneHotEncode            
    "street_name",
    "postal_code",
    "com_name",
    "dep_code",
    "com_code",
    "section_prefix",
    "section",
    "plot_number",
    "volume_number",
    "lot_1",
    "lot_2",
    "lot_3",
    "lot_4",
    "lot_5",
    "property_type_code",
    "property_type",
    "land_nature",
    "land_nature_special",
    "surface_type",
    "com_type",
    "insee_code",

    # date split to transaction_year and transaction_month (categorical variables)
    # already dropped transaction_date
   "transaction_year",
   "transaction_month"

]

# numerical variables to Feature Scale
num_cols = [
    "lot_1_surface",
    "lot_2_surface",
    "lot_3_surface",
    "lot_4_surface",
    "lot_5_surface",
    "lots_count",
    "building_real_surface",
    "main_rooms_count",
    "land_surface",
    "effective_surface",
    "value_per_m2",
    "longitude",
    "latitude"
]

### **Feature Engineering - Categorical**

**All Models require Encoding Categorical Features** 
- OneHotEncoder (nominal)
- OrdinalEncoder (ordinal)
- Handle unknown categories when Encoding
- Avoid label encoding for nominal variables


In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# ---- One-Hot Encode Categorical Features ----
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

### **Feature Scaling - Numeric**

**Scaling can be conditional per model:**

**Required for:** 
- Linear models (Logistic Regression, Ridge, Lasso, ElasticNet)
- Support Vector Machines (SVM)
- K-Nearest Neighbors (KNN)
- Neural Networks
- K-Means / distance-based algorithms

**NOT required for:** 
- Tree-based models (Decision Trees, Random Forest, XGBoost, LightGBM, CatBoost)

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, RobustScaler

# ---- Scale Numeric Features ----
# scaler = StandardScaler()
# StandardScaler scales numerical features to have mean=0 and std=1

scaler = RobustScaler()
# RobustScaler scales numerical features using median and IQR, making it more robust to outliers

## **📚 STEP 3.5 — CHECK TARGET (y) DISTRIBUTION in `y_train` & `y_test`**

In [ ]:
import pandas as pd

# Reload the (possibly overwritten) splits
y_train = pd.read_csv("../data/processed/y_train.csv").values.ravel()
y_test = pd.read_csv("../data/processed/y_test.csv").values.ravel()

print("Train class distribution:")
print(pd.Series(y_train).value_counts())
print(pd.Series(y_train).value_counts(normalize=True))

print("\nTest class distribution:")
print(pd.Series(y_test).value_counts())
print(pd.Series(y_test).value_counts(normalize=True))

## **💦 STEP 4 — Handle DATA LEAKAGE**

Data leakage happens when:
- Information from the Test set influences training
- Or when a feature contains **future information**
- Or when preprocessing is done **before splitting**

It's important to:
- Remove post-event features
- Remove target proxies
- Remove ID-like features
- Check correlation with target

Thus:
- We Drop leakage features
- Ensure no post-target info (variables) exists
- Fit transformations only on training set

**DROP**
- identifier column (ID) and (Names), they are not useful features (high cardinality and not predictive)
- columns that are future information

Note:
- This step is more important than imbalance handling
- This step can be done either before or right after the `STEP 2 - Train/Test SPLIT`.

## **⚖️ STEP 5 — Handle TARGET IMBALANCE (skew + rare values)**